In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col, length

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")

In [0]:
df.columns

In [0]:
COL_RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in COL_RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.limit(5).display()

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df.schema["order_date"].dataType

### Cleaning Date data type that is found to be stored in Integer type

In [0]:
df = (
    df
    .withColumn(
        "order_date",
        F.when(
            (col("order_date") == 0) | (length(col("order_date")) != 8),
            None
        ).otherwise(F.to_date(col("order_date").cast("string"), "yyyyMMdd")
    )
    )
    .withColumn(
        "ship_date",
        F.when(
            (col("ship_date") == 0) | (length(col("ship_date")) != 8),
            None
        ).otherwise(F.to_date(col("ship_date").cast("string"), "yyyyMMdd")
    )
    )
    .withColumn(
        "due_date",
        F.when(
            (col("due_date") == 0) | (length(col("due_date")) != 8),
            None
        ).otherwise(F.to_date(col("due_date").cast("string"), "yyyyMMdd")
    )       
    )
)

In [0]:
df.schema["due_date"].dataType

In [0]:
df_neg_sales = df.filter(col("sales_amount") <= 0)
df_neg_sales.display()

In [0]:
df = (
    df
    .withColumn(
        "sales_amount",
        F.when(
            (col("sales_amount") <= 0) | (col("sales_amount").isNull()), 
            F.when(
                (col("price") > 0) & (col("quantity") > 0),
                col("price") * col("quantity")
            ).otherwise(None)
        ).otherwise(col("sales_amount"))
    )
)

In [0]:
df.filter((col("sales_amount").isNull()) | (col("sales_amount") <= 0)).count()

In [0]:
df_neg_price = df.filter((col("price") <= 0) | (col("price").isNull()))
df_neg_price.display()

In [0]:
df = (
    df
    .withColumn(
        "price",
        F.when(
            (col("price") <= 0) | (col("price").isNull()),
            F.when(
                (col("sales_amount") > 0) & (col("quantity") > 0),
                col("sales_amount") / col("quantity")
            ).otherwise(None)
        ).otherwise(col("price"))
)
)

In [0]:
df.filter((col("price") <= 0) | (col("price").isNull())).count()

In [0]:
df.limit(5).display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

In [0]:
%sql
SELECT * FROM workspace.silver.crm_sales LIMIT(5);